# Bài 05 — Tổ hợp tuyến tính, Span và Basis với NumPy

Tài liệu đi kèm gồm [lý thuyết](notes.md) và [ứng dụng ML](ml-in-practice.md).

## Mục tiêu

- Viết tổ hợp tuyến tính bằng phép nhân ma trận–vector.
- Kiểm tra span, độc lập tuyến tính và rút basis từ tập sinh.
- Giải tọa độ cho một vector và cho cả batch rồi tái tạo dữ liệu.
- Dùng phần dư least squares để phân biệt vector thuộc và ngoài span.
- Thực hành truy xuất embedding bằng cosine và kiểm tra hình học sau khi đổi tọa độ.

Hãy dự đoán shape, số hướng độc lập và sai số tái tạo trước khi chạy từng ví dụ.

## 0. Khởi tạo

Notebook dùng dữ liệu minh họa cố định và chỉ cần NumPy.

In [1]:
import numpy as np

np.set_printoptions(precision=5, suppress=True)

## 1. Tổ hợp tuyến tính là phép nhân ma trận–vector

Mỗi cột của `A` là một vector sinh. Hệ số âm đảo chiều đóng góp tương ứng. Hai cách tính dưới đây cho cùng vector dù biểu thức thứ hai dựa vào broadcasting.

In [2]:
v1 = np.array([1.0, 2.0])
v2 = np.array([2.0, 1.0])
A = np.column_stack([v1, v2])
weights = np.array([3.0, -1.0])
combined = A @ weights

assert A.shape == (2, 2) and weights.shape == (2,)
assert combined.shape == (2,)
np.testing.assert_allclose(combined, [1.0, 5.0])
np.testing.assert_allclose(combined, np.sum(A * weights, axis=1))
print("Tổ hợp:", combined)

Tổ hợp: [1. 5.]


## 2. Từ đường thẳng đến toàn bộ không gian

Hai vector cùng phương chỉ sinh một đường. Hai hướng độc lập trong 3D sinh một mặt phẳng. Cột thứ ba dưới đây bổ sung một hướng ngoài mặt phẳng, làm span thành toàn bộ ℝ³.

In [3]:
u = np.array([1.0, 0.0, 1.0])
v = np.array([0.0, 1.0, 1.0])
vertical = np.array([0.0, 0.0, 1.0])

line = np.column_stack([u, 2.0 * u])
plane = np.column_stack([u, v])
space = np.column_stack([u, v, vertical])

ranks = [np.linalg.matrix_rank(G) for G in (line, plane, space)]
assert ranks == [1, 2, 3]
print("Số chiều của các span:", ranks)
print("Một điểm trên mặt phẳng z = x + y:", plane @ np.array([2.0, -1.0]))

Số chiều của các span: [np.int64(1), np.int64(2), np.int64(3)]
Một điểm trên mặt phẳng z = x + y: [ 2. -1.  1.]


## 3. Một quan hệ phụ thuộc và hai bộ hệ số

Thêm `u + 2*v` vào tập sinh giữ nguyên span. Cộng một bội của quan hệ phụ thuộc vào hệ số hiện có cũng giữ nguyên vector đầu ra.

In [4]:
redundant = np.column_stack([u, v, u + 2.0 * v])
relation = np.array([1.0, 2.0, -1.0])
c1 = np.array([3.0, -1.0, 0.0])
c2 = c1 + relation

np.testing.assert_allclose(redundant @ relation, np.zeros(3))
assert np.linalg.matrix_rank(redundant) < redundant.shape[1]
assert not np.array_equal(c1, c2)
np.testing.assert_allclose(redundant @ c1, redundant @ c2)
print("Hai bộ hệ số:", c1, c2)
print("Cùng vector:", redundant @ c1)

Hai bộ hệ số: [ 3. -1.  0.] [ 4.  1. -1.]
Cùng vector: [ 3. -1.  2.]


## 4. Giải tọa độ và tái tạo một batch

Cột của `B` là các vector basis. Dữ liệu `X` lưu theo hàng, nên `X.T` đưa mỗi vector thành một vế phải. Kết quả được chuyển vị để lưu lại theo hàng.

In [5]:
B = np.array([[1.0, 1.0], [1.0, -1.0]])
X = np.array([[4.0, 2.0], [2.0, 0.0], [0.0, -2.0]])

C = np.linalg.solve(B, X.T).T
X_reconstructed = C @ B.T

assert C.shape == X.shape
np.testing.assert_allclose(C[0], [3.0, 1.0])
np.testing.assert_allclose(X_reconstructed, X, atol=1e-12)
print("Tọa độ theo B:\n", C)

Tọa độ theo B:
 [[ 3.  1.]
 [ 1.  1.]
 [-1.  1.]]


## 5. Kiểm tra thuộc span bằng phần dư

`lstsq` luôn tìm bộ hệ số giảm sai số. Để quyết định vector có thuộc span, tính phần dư trực tiếp và so với ngưỡng theo scale của target. Với mặt phẳng ở đây, hãy đối chiếu kết quả với điều kiện `z = x + y`.

In [6]:
targets = np.array([[2.0, -1.0, 1.0], [2.0, -1.0, 0.0]])
coefficients_t, _, _, _ = np.linalg.lstsq(plane, targets.T, rcond=None)
fits = (plane @ coefficients_t).T
errors = np.linalg.norm(fits - targets, axis=1)
limits = 1e-10 + 1e-8 * np.linalg.norm(targets, axis=1)
in_span = errors <= limits

np.testing.assert_array_equal(in_span, [True, False])
np.testing.assert_allclose(fits[0], targets[0], atol=1e-12)
print("Phần dư:", errors)
print("Thuộc span:", in_span)

Phần dư: [0.      0.57735]
Thuộc span: [ True False]


## 6. Rút basis từ một tập sinh

Duyệt cột gốc và giữ cột làm rank tăng. Ví dụ dùng tolerance cố định cho dữ liệu nhỏ cùng scale. Những hướng gần phụ thuộc có thể được giữ hoặc bỏ tùy tolerance.

In [7]:
generators = np.column_stack([u, 2.0 * u, v, u + v])
selected = []
for j in range(generators.shape[1]):
    candidate = generators[:, selected + [j]]
    if np.linalg.matrix_rank(candidate, tol=1e-10) > len(selected):
        selected.append(j)

selected_basis = generators[:, selected]
generator_coordinates = np.linalg.lstsq(selected_basis, generators, rcond=None)[0]

assert selected == [0, 2]
np.testing.assert_allclose(selected_basis @ generator_coordinates, generators, atol=1e-12)
print("Cột được chọn:", selected)

Cột được chọn: [0, 2]


## 7. Hồi quy tuyến tính với feature dư

Với hai feature phụ thuộc, có thể đổi trọng số mà vẫn giữ nguyên dự đoán. Xem vector dự đoán như tổ hợp các cột của design matrix.

In [8]:
feature = np.array([1.0, 2.0, 3.0])
design = np.column_stack([feature, 2.0 * feature])
w1 = np.array([3.0, 0.0])
w2 = np.array([1.0, 1.0])

assert np.linalg.matrix_rank(design) == 1
np.testing.assert_allclose(design @ w1, design @ w2)
print("Dự đoán chung:", design @ w1)

Dự đoán chung: [3. 6. 9.]


## 8. Truy xuất một từ bằng cosine

Tổ hợp tuyến tính tạo query. Cosine xếp hạng các hàng của bảng. Ví dụ dành một hàng zero để minh họa cách lọc ứng viên, rồi loại ba từ đã dùng để tạo query.

In [9]:
def cosine_scores(embeddings, query, excluded=()):
    query_norm = np.linalg.norm(query)
    if query_norm == 0:
        raise ValueError("Cosine cần query khác zero.")

    row_norms = np.linalg.norm(embeddings, axis=1)
    eligible = row_norms > 0
    eligible[list(excluded)] = False
    if not eligible.any():
        raise ValueError("Cần ít nhất một ứng viên khác zero.")

    scores = np.full(embeddings.shape[0], -np.inf)
    scores[eligible] = (embeddings[eligible] @ query) / (row_norms[eligible] * query_norm)
    return scores


words = np.array(["king", "man", "woman", "queen", "river", "book", "<pad>"])
E = np.array([
    [3.0, 1.0], [2.0, 0.0], [0.0, 2.0], [1.0, 3.0],
    [-1.0, 1.0], [2.0, 2.0], [0.0, 0.0],
])
source_ids = np.array([0, 1, 2])
query_weights = np.array([1.0, -1.0, 1.0])
query = query_weights @ E[source_ids]
scores = cosine_scores(E, query, source_ids)

np.testing.assert_allclose(query, [1.0, 3.0])
assert words[np.argmax(scores)] == "queen"
assert np.isneginf(scores[-1])
print(dict(zip(words, scores)))

{np.str_('king'): np.float64(-inf), np.str_('man'): np.float64(-inf), np.str_('woman'): np.float64(-inf), np.str_('queen'): np.float64(0.9999999999999998), np.str_('river'): np.float64(0.4472135954999579), np.str_('book'): np.float64(0.8944271909999159), np.str_('<pad>'): np.float64(-inf)}


## Bài tập

Có 7 bài tăng dần và một bài tổng hợp. Hãy dự đoán kết quả, điền các dòng `TODO`
rồi chạy ô kiểm tra ngay sau bài làm. Mở phần đáp án khi cần đối chiếu.
Các ô kiểm tra báo tên biến còn thiếu nếu bạn chạy toàn bộ notebook trước khi điền bài.

### Bài 1 — Tổ hợp với hệ số âm

Xếp ba vector sau thành các cột của `ex1_A` rồi tính `ex1_result` bằng `@`.

In [10]:
ex1_v1 = np.array([1.0, 0.0, 2.0])
ex1_v2 = np.array([0.0, 2.0, 1.0])
ex1_v3 = np.array([1.0, 1.0, -1.0])
ex1_weights = np.array([2.0, -1.0, 0.5])

# TODO
# ex1_A = ...
# ex1_result = ...

In [11]:
required = ["ex1_A","ex1_result"]
missing = [name for name in required if name not in globals()]
if missing:
    print("⏳ Còn thiếu:", ", ".join(missing))
else:
    assert ex1_A.shape == (3, 3)
    assert ex1_result.shape == (3,)
    np.testing.assert_allclose(ex1_A[:, 0], ex1_v1)
    np.testing.assert_allclose(ex1_A[:, 1], ex1_v2)
    np.testing.assert_allclose(ex1_A[:, 2], ex1_v3)
    np.testing.assert_allclose(ex1_result, [2.5, -1.5, 2.5])
    print("✅ Đạt — Bài 1 — Tổ hợp với hệ số âm")

⏳ Còn thiếu: ex1_A, ex1_result


<details><summary>Đáp án Bài 1</summary>

```python
ex1_A = np.column_stack([ex1_v1, ex1_v2, ex1_v3])
ex1_result = ex1_A @ ex1_weights
```

</details>

### Bài 2 — Số chiều và khả năng sinh

Tính rank cho từng tập. Tạo danh sách `ex2_independent` và `ex2_spans_r2` gồm một giá trị boolean cho mỗi tập. Tìm tập nào là basis của ℝ².

In [12]:
ex2_sets = [
    np.array([[1.0, 2.0], [2.0, 4.0]]),
    np.array([[1.0, 0.0, 1.0], [0.0, 1.0, 1.0]]),
    np.array([[1.0, 1.0], [0.0, 1.0]]),
]

# TODO
# ex2_ranks = ...
# ex2_independent = ...
# ex2_spans_r2 = ...

In [13]:
required = ["ex2_ranks","ex2_independent","ex2_spans_r2"]
missing = [name for name in required if name not in globals()]
if missing:
    print("⏳ Còn thiếu:", ", ".join(missing))
else:
    np.testing.assert_array_equal(ex2_ranks, [1, 2, 2])
    np.testing.assert_array_equal(ex2_independent, [False, False, True])
    np.testing.assert_array_equal(ex2_spans_r2, [False, True, True])
    print("✅ Đạt — Bài 2 — Số chiều và khả năng sinh")

⏳ Còn thiếu: ex2_ranks, ex2_independent, ex2_spans_r2


<details><summary>Đáp án Bài 2</summary>

```python
ex2_ranks = [np.linalg.matrix_rank(G) for G in ex2_sets]
ex2_independent = [r == G.shape[1] for r, G in zip(ex2_ranks, ex2_sets)]
ex2_spans_r2 = [r == 2 for r in ex2_ranks]
```

Chỉ tập thứ ba vừa độc lập vừa sinh ℝ².

</details>

### Bài 3 — Chứng minh một cột dư

Tìm quan hệ phụ thuộc `ex3_relation` có hệ số cuối bằng `-1`. Dùng quan hệ đó để biểu diễn cột cuối từ hai cột đầu và lưu vào `ex3_rebuilt`.

In [14]:
ex3_A = np.array([[1.0, 0.0, 2.0], [0.0, 1.0, -3.0], [1.0, 1.0, -1.0]])

# TODO
# ex3_relation = ...
# ex3_rebuilt = ...

In [15]:
required = ["ex3_relation","ex3_rebuilt"]
missing = [name for name in required if name not in globals()]
if missing:
    print("⏳ Còn thiếu:", ", ".join(missing))
else:
    assert ex3_relation.shape == (3,)
    assert ex3_relation[-1] == -1.0
    np.testing.assert_allclose(ex3_A @ ex3_relation, np.zeros(3), atol=1e-12)
    np.testing.assert_allclose(ex3_rebuilt, ex3_A[:, 2], atol=1e-12)
    print("✅ Đạt — Bài 3 — Chứng minh một cột dư")

⏳ Còn thiếu: ex3_relation, ex3_rebuilt


<details><summary>Đáp án Bài 3</summary>

```python
ex3_relation = np.array([2.0, -3.0, -1.0])
ex3_rebuilt = ex3_A[:, :2] @ ex3_relation[:2]
```

</details>

### Bài 4 — Tọa độ trong basis không trực chuẩn

Giải tọa độ của cả ba hàng trong một lần gọi `solve`. Tái tạo về hệ chuẩn và so sánh với kết quả dùng chuyển vị để thử lấy tọa độ.

In [16]:
ex4_B = np.array([[2.0, 1.0], [0.0, 1.0]])
ex4_X = np.array([[5.0, 1.0], [-1.0, 3.0], [0.0, 2.0]])

# TODO
# ex4_coords = ...
# ex4_rebuilt = ...
# ex4_transpose_guess = ...

In [17]:
required = ["ex4_coords","ex4_rebuilt","ex4_transpose_guess"]
missing = [name for name in required if name not in globals()]
if missing:
    print("⏳ Còn thiếu:", ", ".join(missing))
else:
    assert ex4_coords.shape == (3, 2)
    np.testing.assert_allclose(ex4_coords, [[2.0, 1.0], [-2.0, 3.0], [-1.0, 2.0]])
    np.testing.assert_allclose(ex4_rebuilt, ex4_X, atol=1e-12)
    np.testing.assert_allclose(ex4_transpose_guess, ex4_X @ ex4_B)
    assert not np.allclose(ex4_transpose_guess, ex4_coords)
    print("✅ Đạt — Bài 4 — Tọa độ trong basis không trực chuẩn")

⏳ Còn thiếu: ex4_coords, ex4_rebuilt, ex4_transpose_guess


<details><summary>Đáp án Bài 4</summary>

```python
ex4_coords = np.linalg.solve(ex4_B, ex4_X.T).T
ex4_rebuilt = ex4_coords @ ex4_B.T
ex4_transpose_guess = ex4_X @ ex4_B
```

</details>

### Bài 5 — Nhận diện vector ngoài span

Dùng `lstsq` cho batch `ex5_targets`. Tính norm phần dư cho từng hàng, rồi tạo mask `ex5_in_span` với ngưỡng `1e-10`.

In [18]:
ex5_basis = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
ex5_targets = np.array([[1.0, 2.0, 3.0], [1.0, 1.0, 3.0], [0.0, 0.0, 0.0]])

# TODO
# ex5_coords_t = ...
# ex5_rebuilt = ...
# ex5_errors = ...
# ex5_in_span = ...

In [19]:
required = ["ex5_coords_t","ex5_rebuilt","ex5_errors","ex5_in_span"]
missing = [name for name in required if name not in globals()]
if missing:
    print("⏳ Còn thiếu:", ", ".join(missing))
else:
    assert ex5_coords_t.shape == (2, 3)
    assert ex5_rebuilt.shape == (3, 3)
    np.testing.assert_allclose(ex5_errors, [0.0, 1.0 / np.sqrt(3.0), 0.0], atol=1e-12)
    np.testing.assert_array_equal(ex5_in_span, [True, False, True])
    print("✅ Đạt — Bài 5 — Nhận diện vector ngoài span")

⏳ Còn thiếu: ex5_coords_t, ex5_rebuilt, ex5_errors, ex5_in_span


<details><summary>Đáp án Bài 5</summary>

```python
ex5_coords_t = np.linalg.lstsq(ex5_basis, ex5_targets.T, rcond=None)[0]
ex5_rebuilt = (ex5_basis @ ex5_coords_t).T
ex5_errors = np.linalg.norm(ex5_rebuilt - ex5_targets, axis=1)
ex5_in_span = ex5_errors <= 1e-10
```

</details>

### Bài 6 — Chọn basis từ các cột gốc

Duyệt cột theo thứ tự, dùng rank với `tol=1e-10`, rồi tái tạo mọi cột từ basis tìm được.

In [20]:
ex6_G = np.array([
    [0.0, 1.0, 2.0, 0.0, 1.0],
    [0.0, 0.0, 0.0, 1.0, 1.0],
    [0.0, 1.0, 2.0, 1.0, 2.0],
])

# TODO
# ex6_indices = ...
# ex6_basis = ...
# ex6_rebuilt = ...

In [21]:
required = ["ex6_indices","ex6_basis","ex6_rebuilt"]
missing = [name for name in required if name not in globals()]
if missing:
    print("⏳ Còn thiếu:", ", ".join(missing))
else:
    np.testing.assert_array_equal(ex6_indices, [1, 3])
    assert ex6_basis.shape == (3, 2)
    assert np.linalg.matrix_rank(ex6_basis) == 2
    np.testing.assert_allclose(ex6_basis, ex6_G[:, [1, 3]])
    np.testing.assert_allclose(ex6_rebuilt, ex6_G, atol=1e-12)
    print("✅ Đạt — Bài 6 — Chọn basis từ các cột gốc")

⏳ Còn thiếu: ex6_indices, ex6_basis, ex6_rebuilt


<details><summary>Đáp án Bài 6</summary>

```python
ex6_indices = []
for j in range(ex6_G.shape[1]):
    candidate = ex6_G[:, ex6_indices + [j]]
    if np.linalg.matrix_rank(candidate, tol=1e-10) > len(ex6_indices):
        ex6_indices.append(j)

ex6_basis = ex6_G[:, ex6_indices]
coords = np.linalg.lstsq(ex6_basis, ex6_G, rcond=None)[0]
ex6_rebuilt = ex6_basis @ coords
```

</details>

### Bài 7 — Cosine trong hai hệ tọa độ

Tính cosine giữa hai hàng của `ex7_C`. Tái tạo chúng về 3D rồi tính lại cosine. Có thể dùng `cosine_scores` với hàng thứ nhất làm query và hàng thứ hai làm ứng viên.

In [22]:
ex7_basis = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
ex7_C = np.eye(2)

# TODO
# ex7_original = ...
# ex7_cos_coords = ...
# ex7_cos_original = ...

In [23]:
required = ["ex7_original","ex7_cos_coords","ex7_cos_original"]
missing = [name for name in required if name not in globals()]
if missing:
    print("⏳ Còn thiếu:", ", ".join(missing))
else:
    assert ex7_original.shape == (2, 3)
    np.testing.assert_allclose(ex7_original, [[1.0, 0.0, 1.0], [0.0, 1.0, 1.0]])
    np.testing.assert_allclose(ex7_cos_coords, 0.0, atol=1e-12)
    np.testing.assert_allclose(ex7_cos_original, 0.5)
    print("✅ Đạt — Bài 7 — Cosine trong hai hệ tọa độ")

⏳ Còn thiếu: ex7_original, ex7_cos_coords, ex7_cos_original


<details><summary>Đáp án Bài 7</summary>

```python
ex7_original = ex7_C @ ex7_basis.T
ex7_cos_coords = cosine_scores(ex7_C[1:], ex7_C[0])[0]
ex7_cos_original = cosine_scores(ex7_original[1:], ex7_original[0])[0]
```

</details>

## Bài tổng hợp — Nén và truy xuất embedding

Bảng `cap_embeddings` có bảy từ, ba tọa độ mỗi từ và một hàng padding bằng zero.
Các hàng nằm trên mặt phẳng `z = x + y`. Hãy thử tái tạo từ tọa độ 2D rồi so sánh
với kết quả truy xuất trên bảng gốc.

1. Dùng `lstsq` để mã hóa bảng theo `cap_basis`. Lưu tọa độ theo hàng.
2. Tái tạo bảng 3D và tính norm phần dư trên từng hàng.
3. Tạo truy vấn `king - man + woman` trên tọa độ 2D rồi tái tạo truy vấn về 3D.
4. Chấm điểm cosine trên các hàng đã tái tạo và loại ba từ nguồn. Làm tương tự với bảng gốc để đối chiếu.
5. Thử mã hóa `cap_unseen`. Đo phần dư và giải thích vì sao hàng mới cho kết quả khác bảng minh họa.

### Nén và truy xuất embedding

Dùng lại hàm `cosine_scores` ở Phần 8.

In [24]:
cap_embeddings = np.column_stack([E, E.sum(axis=1)])
cap_basis = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
cap_unseen = np.array([1.0, 1.0, 3.0])

# TODO 1–2: mã hóa bảng, tái tạo và kiểm tra phần dư
# cap_coordinates = ...
# cap_reconstructed = ...
# cap_errors = ...

# TODO 3–4: query và truy xuất
# cap_query_coordinates = ...
# cap_query = ...
# cap_scores = ...
# cap_original_query = ...
# cap_original_scores = ...
# cap_answer = ...

# TODO 5: sai số của một hàng ngoài mặt phẳng
# cap_unseen_coordinates = ...
# cap_unseen_error = ...

In [25]:
required = ["cap_coordinates","cap_reconstructed","cap_errors","cap_query_coordinates","cap_query","cap_scores","cap_original_query","cap_original_scores","cap_answer","cap_unseen_coordinates","cap_unseen_error"]
missing = [name for name in required if name not in globals()]
if missing:
    print("⏳ Còn thiếu:", ", ".join(missing))
else:
    assert cap_coordinates.shape == (len(words), 2)
    assert cap_reconstructed.shape == cap_embeddings.shape
    assert cap_errors.shape == (len(words),)
    np.testing.assert_allclose(cap_reconstructed, cap_embeddings, atol=1e-12)
    np.testing.assert_allclose(cap_errors, 0.0, atol=1e-12)
    np.testing.assert_allclose(cap_query_coordinates, [1.0, 3.0], atol=1e-12)
    np.testing.assert_allclose(cap_original_query, [1.0, 3.0, 4.0], atol=1e-12)
    np.testing.assert_allclose(cap_query, cap_original_query, atol=1e-12)

    assert cap_scores.shape == cap_original_scores.shape == (len(words),)
    assert np.isneginf(cap_scores[source_ids]).all()
    assert np.isneginf(cap_scores[-1])
    np.testing.assert_allclose(cap_original_scores, cosine_scores(cap_embeddings, cap_original_query, source_ids))
    np.testing.assert_allclose(cap_scores, cap_original_scores, atol=1e-12)
    assert cap_answer == words[np.argmax(cap_scores)] == "queen"
    np.testing.assert_allclose(cap_unseen_coordinates, [4.0 / 3.0, 4.0 / 3.0])
    np.testing.assert_allclose(cap_unseen_error, 1.0 / np.sqrt(3.0))

    print("Từ được truy xuất:", cap_answer)
    print("Sai số tái tạo lớn nhất của bảng:", cap_errors.max())
    print("Sai số tái tạo vector mới:", cap_unseen_error)
    print("✅ Đạt — Nén và truy xuất embedding")

⏳ Còn thiếu: cap_coordinates, cap_reconstructed, cap_errors, cap_query_coordinates, cap_query, cap_scores, cap_original_query, cap_original_scores, cap_answer, cap_unseen_coordinates, cap_unseen_error


<details><summary>Đáp án bài tổng hợp</summary>

```python
coords_t = np.linalg.lstsq(cap_basis, cap_embeddings.T, rcond=None)[0]
cap_coordinates = coords_t.T
cap_reconstructed = cap_coordinates @ cap_basis.T
cap_errors = np.linalg.norm(cap_reconstructed - cap_embeddings, axis=1)

cap_query_coordinates = query_weights @ cap_coordinates[source_ids]
cap_query = cap_basis @ cap_query_coordinates
cap_scores = cosine_scores(cap_reconstructed, cap_query, source_ids)

cap_original_query = query_weights @ cap_embeddings[source_ids]
cap_original_scores = cosine_scores(cap_embeddings, cap_original_query, source_ids)
cap_answer = words[np.argmax(cap_scores)]

cap_unseen_coordinates = np.linalg.lstsq(cap_basis, cap_unseen, rcond=None)[0]
cap_unseen_error = np.linalg.norm(cap_basis @ cap_unseen_coordinates - cap_unseen)
```

Vector mới nằm ngoài mặt phẳng vì tọa độ thứ ba bằng 3, còn tổng hai tọa độ đầu bằng 2.

</details>

## Tự kiểm tra nhanh

1. Với `A.shape == (4, 3)`, `A @ c` cần `c` có shape nào?
2. Ba vector trong ℝ² có thể sinh cả ℝ² và phụ thuộc tuyến tính cùng lúc không?
3. Vì sao giữ thêm một vector cùng phương với vector có sẵn không làm span lớn lên?
4. Rank của một basis cho mặt phẳng qua gốc trong ℝ³ là bao nhiêu?
5. Vì sao phải tính phần dư sau khi gọi `lstsq` để kiểm tra thuộc span?
6. Khi nào chuyển vị của ma trận basis cho phép lấy tọa độ trực tiếp?
7. Vì sao ví dụ embedding tính cosine sau khi tái tạo về 3D?

## Tổng kết

- Tổ hợp tuyến tính `A @ c` chọn một vector trong span của các cột của `A`.
- Basis gồm các hướng độc lập sinh đúng không gian cần biểu diễn. Mỗi vector trong span có một bộ tọa độ duy nhất theo basis đã chọn.
- `solve` phù hợp cho basis vuông khả nghịch. `lstsq` và phần dư giúp kiểm tra thuộc span trong trường hợp tổng quát.
- Trong ML, cùng một phép kiểm tra tái tạo giúp phân biệt dữ liệu có biểu diễn thấp chiều chính xác với dữ liệu cần xấp xỉ.
- Một basis không trực chuẩn có thể thay đổi dot product và cosine khi tính trực tiếp trên tọa độ.

Sang Bài 06, ta sẽ dùng rank để nghiên cứu số chiều của range, row space và null space.